# Lab Work - 4.8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error


## Q1 - Code It

In [ ]:
X = np.array([[1],[2],[3],[4],[5]])
y = np.array([2.1,4.9,5.8,8.2,9.7])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

linear_svr = SVR(kernel='linear', C=1.0, epsilon=0.5)
linear_svr.fit(X_train_scaled, y_train)

train_pred = linear_svr.predict(X_train_scaled)
test_pred = linear_svr.predict(X_test_scaled)

print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))


In [ ]:
print("Support Vectors:")
print(linear_svr.support_vectors_)

print("\nSupport Vector Indices:")
print(linear_svr.support_)

print("\nDual Coefficients:")
print(linear_svr.dual_coef_)


In [ ]:
rbf_svr = SVR(kernel='rbf', C=1.0, epsilon=0.5, gamma='scale')
rbf_svr.fit(X_train_scaled, y_train)

print("Linear Train MSE:", mean_squared_error(y_train, train_pred))
print("Linear Test MSE:", mean_squared_error(y_test, linear_svr.predict(X_test_scaled)))

print("RBF Train MSE:", mean_squared_error(y_train, rbf_svr.predict(X_train_scaled)))
print("RBF Test MSE:", mean_squared_error(y_test, rbf_svr.predict(X_test_scaled)))


## Q2 - Kernel Comparison

In [ ]:
kernels = {
    'linear': SVR(kernel='linear', C=10, epsilon=0.1),
    'poly': SVR(kernel='poly', degree=3, C=10, epsilon=0.1),
    'rbf': SVR(kernel='rbf', gamma='scale', C=10, epsilon=0.1)
}

for name, model in kernels.items():
    scores = -cross_val_score(
        model,
        scaler.fit_transform(X),
        y,
        cv=5,
        scoring='neg_mean_squared_error'
    )
    print(name, "Mean CV MSE =", scores.mean(), "Std =", scores.std())


In [ ]:
param_grid = {
    'C':[0.1,1,10,100],
    'epsilon':[0.01,0.1,0.5,1.0]
}

grid = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

grid.fit(scaler.fit_transform(X), y)

print("Best Params:", grid.best_params_)
print("Best Score:", -grid.best_score_)


In [ ]:
best_C = grid.best_params_['C']
best_eps = grid.best_params_['epsilon']

gammas = [0.01,0.1,1.0,10]

results = []

for g in gammas:
    model = SVR(
        kernel='rbf',
        C=best_C,
        epsilon=best_eps,
        gamma=g
    )

    mse = -cross_val_score(
        model,
        scaler.fit_transform(X),
        y,
        cv=5,
        scoring='neg_mean_squared_error'
    ).mean()

    results.append(mse)

plt.plot(gammas, results, marker='o')
plt.xscale('log')
plt.xlabel("Gamma")
plt.ylabel("CV MSE")
plt.title("Gamma Sweep")
plt.show()


## Q3 - Visualizations

In [ ]:
X_scaled = scaler.fit_transform(X)

model = SVR(kernel='linear', C=1.0, epsilon=0.5)
model.fit(X_scaled, y)

pred = model.predict(X_scaled)

plt.figure(figsize=(8,5))
plt.scatter(X, y, label='Data')
plt.plot(X, pred, label='SVR Fit')

plt.fill_between(
    X.flatten(),
    pred-0.5,
    pred+0.5,
    alpha=0.2
)

plt.scatter(
    X[model.support_],
    y[model.support_],
    s=120,
    marker='s',
    label='Support Vectors'
)

plt.legend()
plt.show()


In [ ]:
C_values = [0.01,1,100]

plt.figure(figsize=(8,5))

for c in C_values:
    model = SVR(kernel='rbf', C=c, epsilon=0.1)
    model.fit(X_scaled, y)
    plt.plot(X, model.predict(X_scaled), label=f'C={c}')

plt.scatter(X,y)
plt.legend()
plt.show()


In [ ]:
eps_values = [0.0,0.5,1.5]

plt.figure(figsize=(8,5))

for e in eps_values:
    model = SVR(kernel='rbf', C=10, epsilon=e)
    model.fit(X_scaled,y)
    plt.plot(X, model.predict(X_scaled), label=f'eps={e}')

plt.scatter(X,y)
plt.legend()
plt.show()


## Q4 - Deep Intuition


### Q4.1
SVR(kernel='rbf', C=1000, epsilon=0.001) with Train MSE≈0 and very high Test MSE indicates severe overfitting.

Suggested fixes:
1. Reduce C
2. Increase epsilon
4. Reduce gamma

### Q4.2
SVR(kernel='linear', C=0.001, epsilon=2.0) with high train and test error indicates underfitting.

Suggested fixes:
1. Increase C
2. Reduce epsilon
4. Try RBF kernel

### Q4.3
SVR:
- Can be linear or non-linear
- Uses support vectors
- Good for small/medium datasets

Random Forest:
- Non-parametric ensemble
- Handles non-linearity naturally
- Robust to outliers

### Q4.4
SVR requires feature scaling because optimization depends on distances and dot products. Tree-based methods split on feature thresholds and are scale-invariant.
